# WoofDecoder — 41 Scene Context Training
**Owner:** Elijah Ha  
**Milestone:** October 30, 2026 — train and evaluate environmental scene cues on real annotations.

**This notebook starts from existing prepared RGB images** and trains a small classification head on frozen pretrained CLIP embeddings.

This notebook does not modify `40_scene_zero_shot.ipynb`. It uses train/val only; the test split remains untouched.

## 1. Setup
Use the project environment with the ml extra installed and select that interpreter in Jupyter. The reusable model code is in src/woofdecoder/branches/scene.py.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import CLIPModel, CLIPProcessor

# Find the project, including the workspace layout used by last week's notebook.
roots = [Path.cwd(), *Path.cwd().parents]
REPO_DIR = next((c for root in roots
                 for c in (root, root / "src/Project/dog-vocal-interpreter")
                 if (c / "src/woofdecoder/taxonomy.py").is_file()), None)
if REPO_DIR is None:
    raise RuntimeError("Open this notebook from the WoofDecoder repo or CSSE416 workspace.")
sys.path.insert(0, str(REPO_DIR / "src"))

from woofdata.loader import load_sample
from woofdata.path import load_data_dir_from_env_file
from woofdata.tasks import task_rows
from woofdecoder.branches.scene import (
    SCENE_CUES,
    SceneCueHead,
    encode_frames,
    extract_object_cues,
    predict_scene_cues,
)

load_data_dir_from_env_file(REPO_DIR / ".env")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/clip-vit-base-patch32"

# Begin with a reproducible subset; set both to None for the complete split.
MAX_TRAIN = 1200
MAX_VAL = 300
ENCODER_BATCH_SIZE = 16   # lower to 4 or 8 if GPU memory is tight
TRAIN_BATCH_SIZE = 64
EPOCHS = 25
LR = 0.001
PATIENCE = 5

CHECKPOINT_DIR = REPO_DIR / "models" / "scene"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo:", REPO_DIR)
print("Device:", DEVICE)
print("Cue labels:", SCENE_CUES)

## 2. Load prepared images
The scene_objects loader returns RGB images, boxes, and unified category IDs (1 dog, 2 person, 3 leash, 4 bowl, 5 cat). Prepare rf_leash and rf_bowl before running. Video frame extraction is only needed later for bark-segment inference.

In [ ]:
train_rows_all = task_rows("scene_objects", "train")
val_rows_all = task_rows("scene_objects", "val", eval_only=True)
if train_rows_all.empty or val_rows_all.empty:
    raise RuntimeError("Prepare rf_leash and rf_bowl first; scene train/val rows are missing.")

# Check the full splits before sampling, including augmented/recording groups.
for column in ("clip_id", "leakage_group"):
    if column in train_rows_all and column in val_rows_all:
        overlap = set(train_rows_all[column].dropna()) & set(val_rows_all[column].dropna())
        if overlap:
            raise ValueError(f"Train/val overlap in {column}: {len(overlap)} groups")

probe = load_sample("scene_objects", train_rows_all.iloc[0])
print("Images:", len(train_rows_all), "train,", len(val_rows_all), "validation")
print("First object target:", extract_object_cues(probe))
print(train_rows_all["source"].value_counts())
missing_sources = {"rf_leash", "rf_bowl"} - set(train_rows_all["source"])
if missing_sources:
    print("Not prepared in this manifest:", sorted(missing_sources))


## 3 — Check splits and choose a reproducible subset
This experiment uses the team's prepared splits, not a random frame-level re-split. If a source contains near-duplicate augmented images across splits, that must be checked separately. We verify exact clip-ID overlap here.

In [ ]:
def limited(rows, limit):
    if limit is None or limit >= len(rows):
        return rows.reset_index(drop=True)
    return rows.sample(n=limit, random_state=SEED).reset_index(drop=True)

train_rows = limited(train_rows_all, MAX_TRAIN)
val_rows = limited(val_rows_all, MAX_VAL)
train_ids = set(train_rows["clip_id"].astype(str))
val_ids = set(val_rows["clip_id"].astype(str))
assert not train_ids.intersection(val_ids), "Train/val clip IDs overlap!"
print(f"Experiment: {len(train_rows)} train, {len(val_rows)} val | 0 exact clip-ID overlaps")
print("Training by source:")
print(train_rows["clip_id"].astype(str).str.split("/").str[0].value_counts().to_string())

## 4. Encode images once
Freeze CLIP and compute embeddings in batches. Reuse X_train and X_val while experimenting with the small head; rerun this cell after changing prepared data. This avoids a disk cache silently retaining old annotations.

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
for parameter in clip_model.parameters():
    parameter.requires_grad_(False)
print("Frozen image encoder loaded.")

def get_embeddings(rows):
    features_list, targets_list = [], []
    for start in range(0, len(rows), ENCODER_BATCH_SIZE):
        mini = rows.iloc[start:start + ENCODER_BATCH_SIZE]
        images, labels = [], []
        for _, row in mini.iterrows():
            sample = load_sample("scene_objects", row)
            label = extract_object_cues(sample)
            img = np.asarray(sample.image)
            if img.ndim != 3 or img.shape[-1] != 3:
                raise ValueError(f"Non-RGB scene image for {row['clip_id']}: {img.shape}")
            images.append(img)
            labels.append(label)
        features_list.append(encode_frames(images, clip_model, processor, DEVICE,
                                           batch_size=ENCODER_BATCH_SIZE))
        targets_list.append(torch.from_numpy(np.stack(labels)))
        if (start // ENCODER_BATCH_SIZE) % 20 == 0:
            print(f"Encoded {min(start + len(mini), len(rows))}/{len(rows)}")
    result = {
        "features": torch.cat(features_list).float(),
        "targets": torch.cat(targets_list).float(),
    }
    return result["features"], result["targets"]

X_train, Y_train = get_embeddings(train_rows)
X_val, Y_val = get_embeddings(val_rows)
assert X_train.ndim == 2 and X_train.shape[0] == len(train_rows)
assert Y_train.shape == (len(train_rows), len(SCENE_CUES))
assert X_val.shape[1] == X_train.shape[1]
print("Train embeddings / targets:", tuple(X_train.shape), tuple(Y_train.shape))
print("Validation embeddings / targets:", tuple(X_val.shape), tuple(Y_val.shape))

## 5 — Inspect class balance, including rare objects
Labels are multi-label: one image may contain several objects. Report the number of positives for each cue in train and validation. A cue with zero training positives cannot be learned; a cue with zero validation positives cannot receive a meaningful recall score.

In [ ]:
balance = pd.DataFrame({
    "cue": SCENE_CUES,
    "train_positive": Y_train.sum(dim=0).int().tolist(),
    "train_total": len(Y_train),
    "val_positive": Y_val.sum(dim=0).int().tolist(),
    "val_total": len(Y_val),
})
display(balance)
if (Y_train.sum(0) == 0).any():
    print("WARNING: one or more cues have no positive training examples in this subset.")
if (Y_val.sum(0) == 0).any():
    print("WARNING: one or more cues have no positive validation examples; report support.")

## 6 — Evaluate with multi-label metrics
Use sigmoid + threshold 0.5 for each cue (not softmax: several objects can coexist). Report micro-F1, macro-F1 for supported classes, exact-match accuracy, and per-cue precision/recall/F1. Compare the trained head with a training-set majority baseline.

This is **object-cue recognition performance**, not behavioral/bark-context accuracy.

In [ ]:
def evaluate_probs(y_true, probs, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int32)
    preds = (np.asarray(probs) >= threshold).astype(np.int32)
    p, r, f, support = precision_recall_fscore_support(
        y_true, preds, average=None, zero_division=0
    )
    # sklearn supports multi-label per-column scores; support may need explicit computation.
    pos_support = y_true.sum(axis=0)
    supported = pos_support > 0
    details = pd.DataFrame({"cue": SCENE_CUES, "support": pos_support,
                            "precision": p, "recall": r, "F1": f})
    metrics = {
        "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
        "macro_f1_supported": float(f[supported].mean()) if supported.any() else float("nan"),
        "exact_match_accuracy": accuracy_score(y_true, preds),
    }
    return metrics, details

majority_prediction = (Y_train.mean(0).numpy() >= 0.5).astype(float)
majority_probs = np.tile(majority_prediction, (len(Y_val), 1))
majority_metrics, _ = evaluate_probs(Y_val.numpy(), majority_probs)
print("Train-set majority baseline:", majority_metrics)

## 7 — Train the scene-context head
`BCEWithLogitsLoss` is appropriate for independent, possibly co-occurring object labels. Use moderate class weights for rare positives. Early stop on validation macro-F1 (supported cues only). Save the best checkpoint for inference.

In [ ]:
train_set = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_set, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), num_workers=0)

head = SceneCueHead(embedding_dim=X_train.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=0.01)
pos_count = Y_train.sum(dim=0)
neg_count = len(Y_train) - pos_count
pos_weight = torch.where(pos_count > 0, neg_count / pos_count.clamp(min=1),
                         torch.ones_like(pos_count)).clamp(1, 20).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_score = -1.0
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    head.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = head(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    head.eval()
    with torch.inference_mode():
        val_logits = head(X_val.to(DEVICE))
        val_loss = float(loss_fn(val_logits, Y_val.to(DEVICE)).item())
        val_probs = val_logits.sigmoid().cpu().numpy()
    metrics, _ = evaluate_probs(Y_val.numpy(), val_probs)
    row = {"epoch": epoch, "train_loss": total_loss / len(train_set),
           "val_loss": val_loss, **metrics}
    history.append(row)
    print(f"Epoch {epoch:02d}: train={row['train_loss']:.4f} "
          f"val={val_loss:.4f} micro-F1={metrics['micro_f1']:.3f} "
          f"macro-F1={metrics['macro_f1_supported']:.3f}")

    score = metrics["macro_f1_supported"]
    if np.isfinite(score) and score > best_score + 1e-5:
        best_score = score
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Stopped after", PATIENCE, "epochs without improvement")
            break

if best_state is None:
    raise RuntimeError("No valid validation F1; inspect class support before training.")
head.load_state_dict(best_state)
CHECKPOINT = CHECKPOINT_DIR / "scene_cue_head.pt"
torch.save({"state_dict": best_state, "model_id": MODEL_ID,
            "cue_names": list(SCENE_CUES), "embedding_dim": X_train.shape[1],
            "hidden_dim": 256, "dropout": 0.2,
            "train_rows": len(X_train), "val_rows": len(X_val), "seed": SEED}, CHECKPOINT)
print("Saved best checkpoint:", CHECKPOINT)
display(pd.DataFrame(history))

## 8 — Results: trained head vs majority baseline
Report the **actual** validation results, not the two prompt-match scores from last week's zero-shot notebook. If performance is poor, that is still useful: it shows the limits of training scene cues with the available data.

In [ ]:
head.eval()
with torch.inference_mode():
    val_probs = head(X_val.to(DEVICE)).sigmoid().cpu().numpy()
trained_metrics, per_cue_metrics = evaluate_probs(Y_val.numpy(), val_probs)
comparison = pd.DataFrame([
    {"model": "Training majority baseline", **majority_metrics},
    {"model": "Frozen CLIP + trained MLP", **trained_metrics},
])
display(comparison)
display(per_cue_metrics)
print("Best validation macro-F1 (supported cues):", round(best_score, 4))
print("Important: THESE ARE OBJECT-CUE METRICS, NOT BARKING-REASON ACCURACY.")

## 9 — Inference: preprocessed frames → scene features
The preprocessing team will supply the frames (PIL RGB or H×W×3 NumPy arrays). We do not extract them here. The returned cue probabilities have order `SCENE_CUES`, *not* `woofdecoder.taxonomy.CATEGORIES`. The 512-dimensional segment embedding can later be used in a supervised fusion head trained on properly labeled bark-context clips.

In [ ]:
# Smoke inference from one existing prepared image. Replace demo_frames with
# the frame list returned by the preprocessing team's pipeline when integrating.
demo_sample = load_sample("scene_objects", val_rows.iloc[0])
demo_frames = [demo_sample.image]
result = predict_scene_cues(demo_frames, clip_model, processor, head, DEVICE)
print("Cue names:", result["cue_names"])
print("Frame cues:", result["frame_cues"].shape)
print("Segment cues:", result["segment_cues"].shape)
print("Segment embedding:", result["segment_embedding"].shape)
display(pd.DataFrame({"scene_cue": result["cue_names"],
                      "relative_signal": result["segment_cues"]}))
assert result["frame_cues"].shape == (len(demo_frames), len(SCENE_CUES))
assert result["segment_cues"].shape == (len(SCENE_CUES),)